In [1]:
# reprompt normal
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("to_reprompt_normal.tsv", sep="\t", dtype=str)

# df = pd.concat(dfs, ignore_index=True)  # appends; duplicates are kept
df["max_points"] = df["max_points"].str.replace(",", ".").astype(float)
df["human_score"] = df["human_score"].str.replace(",", ".").astype(float)
df["error_type"] = df["error_type"].astype(int)
df["rueckfrage"] = df["rueckfrage"].astype(int)

# check for column names: question_text, student_answer, gold_answer, maximal_score, human_score, human_feedback
df.head()

,Unnamed: 0,id,question_de,answer_de,max_points,keywords,points_z,student_answer,transkript_stt_model,error_type,...,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28
0,3,3,Erkläre das Bias-Variance-Trade-off.,Das Bias-Variance-Trade-off ist der Kompromiss...,6.0,Bias-Variance-Trade-off; Bias; Variance; Unter...,9.860.485.948.630.900,Dieser Trade Off ist der Kompromiss zwischen d...,Dieser Trade Off ist der Kompromiss zwischen d...,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5,5,Wie gehst du mit fehlenden Daten in einem Date...,Fehlende Daten können durch Imputationsverfahr...,6.0,Imputation; Mittelwert; Median; Modus; prädikt...,9.860.485.948.630.900,"So dies möglich erscheint, werden mit der Hilf...","So dies möglich erscheint, werden mit der Hilf...",1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,14,14,Was ist der Unterschied zwischen Klassifikatio...,Die Klassifikation ist eine überwachte Lernauf...,4.0,Klassifikation; Überwachtes Lernen; kategorial...,-11.344.860.177.457.000,"Beides sind überwachte Lernaufgaben, wobei ers...","Beides sind überwachte Lernaufgaben, wobei ers...",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,22,22,Was ist Regularisierung in neuronalen Netzwerken?,Regularisierung in neuronalen Netzwerken beinh...,6.0,Regularisierung; neuronale Netzwerke; Verlustf...,9.860.485.948.630.900,Regularisierung in neuronalen Netzwerken beinh...,Regularisierung in neuronalen Netzwerken beinh...,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,28,28,Was ist Deep Learning?,Deep Learning ist ein Untergebiet des Machine ...,5.0,Untergebiet; Machine Learning; künstliche neur...,-7.421.871.144.130.820,Machine Learning ist ein Untergebiet von Deep ...,Machine Learning ist ein Untergebiet von Deep ...,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
from llm_handler import LLMHandler
import json

# TODO: Write prompt for evaluation
prompt = """
Du bekommst eine Prüfungsfrage, die Antwort eines Studenten darauf, die Musterlösung zu der Frage und die maximal erreichbare Punktzahl für eine richtige Antwort.
Deine Aufgabe ist es, die Antwort des Studenten mit der Musterlösung zu vergleichen und diese zu bewerten.

Dabei sollst du dich an folgendem Leitfaden orientieren:
1) Hat der Student den Sachverhalt fachlich korrekt dargestellt, ohne wesentliche Fehler oder falsche Zusammenhänge?
2) Verwendet der Student die relevanten Schlüsselbegriffe korrekt und im richtigen Kontext? Schau dabei auf folgende Liste der wichtigsten Fachbegriffe: {keywords}.
3) Geht der Student auf alle wesentlichen Aspekte der Fragestellung ein oder bleiben zentrale Punkte unbeantwortet?
4) Werden die angesprochenen Konzepte klar voneinander unterschieden und nicht miteinander vermischt?
5) Ist die Antwort logisch aufgebaut, nachvollziehbar formuliert und für den Prüfer gut verständlich?
6) Soll vom Prüfer eine Rückfrage gestellt werden, um auf Lücken zu prüfen?

Deine Ausgabe soll aus folgenden Elementen bestehen:
- **llm_feedback**: In diesem Feld schreibst du die Bewertung zur Antwort des Studenten. Dabei gehst du auf jeden der oben genannten Punkte ein und formulierst anschließend einen Fließtext daraus. Beschränke dich bei dem Fließtext auf maximal 4-5 Sätze.
- **llm_rating**: Hier vergibst du die Punktzahl, die deiner Meinung nach die Antwort des Studenten wiederspiegelt. Du kannst die Punkte in 0,5 Schritten vergeben. Die maximal vergebbare Punktzahl ist {max_score}.
- **rueckfrage**: Du füllst dieses Feld mit einem Integer aus. Dieser Integer kann 0, 1, 2 oder 3 sein. 0 bedeutet, dass keine Rückfrage notwendig ist. 1 bedeutet, dass eine Rückfrage notwendig ist, die sich auf nicht genannte Fach- bzw. Schlüsselbegriffe bezieht. 2 bedeutet, dass eine Rückfrage gestellt wird und sich auf eine fehlende Teilantwort bezieht. 3 bedeutet, dass sich die gestellte Rückfrage auf einem Teil der falschen Antwort bezieht.

Frage:
{question}

Studentenantwort:
{student_answer}

Musterlösung:
{correct_answer}

Für diese Frage gibt es maximal {max_score} Punkte.

<Antwortformat>
```json
{{
"llm_feedback": "<Hier antwortest du auf die Antwort des Studenten und gibts ihm Feedback entsprechend der Analysepunkte. Die Antwort ist ein Text, es gibt keine JSON-Struktur>",
"llm_rating": "<Gesamtrating, maximale Punkte: {max_score}, gerundet auf eine Nachkommastelle>",
"rueckfrage": "<0 | 1 | 2 | 3>"
}}```
"""

In [4]:
import time

def _results_current_model(prompt: str, df: pd.DataFrame) -> pd.DataFrame:
  llm_results = []
  for _, row in df.iterrows():
    llm = LLMHandler(model=row["model"])
    prompt_filled = prompt.format(
      question=row['question_de'],
      student_answer=row['student_answer'],
      keywords=row["keywords"],
      correct_answer=row['answer_de'],
      max_score=row['max_points']
    )
    while True:
      try:
        print(f"Try processing row {_}")
        answer = llm.call_llm(prompt=prompt_filled)
        if not isinstance(answer, dict):
          raise ValueError("LLM response is not a dictionary")
        answer["question"] = row['question_de']
        answer["student_answer"] = row['student_answer']
        answer["correct_answer"] = row['answer_de']
        answer["keywords"] = row["keywords"]
        answer["max_score"] = row['max_points']
        answer["human_score"] = row['human_score']
        answer["human_feedback"] = row['human_feedback']
        answer["rueckfrage_human"] = row["rueckfrage"]
        answer["model"] = row["model"]
        llm_results.append(answer)
        with open("checkpoint.json", "a", encoding="utf-8") as f:
          f.write(json.dumps(answer, ensure_ascii=False) + "\n")
        break
      except Exception as e:
        print(f"Row {_}: retrying due to error: {e}")
        time.sleep(10)
  return pd.DataFrame(llm_results)

In [5]:

final_df = _results_current_model(
    prompt=prompt,
    df=df
)
final_df.to_csv("reprompt_results_normal_third_try.csv", index=False)

Try processing row 0
Try processing row 1
Try processing row 2
Try processing row 3
Try processing row 4
Try processing row 5
Try processing row 6
Try processing row 7
Try processing row 8
Try processing row 9
Try processing row 10
Try processing row 11
Try processing row 12
Try processing row 13
Try processing row 14
Try processing row 15
Try processing row 16
Try processing row 17
Try processing row 18
Try processing row 19
Try processing row 20
Try processing row 21
Try processing row 22
Try processing row 23
Try processing row 24
Try processing row 25
Try processing row 26
Try processing row 27
Try processing row 28
Row 28: retrying due to error: Expecting ',' delimiter: line 3 column 18 (char 761)
Try processing row 28
Try processing row 29
Try processing row 30
Try processing row 31
Row 31: retrying due to error: Expecting value: line 1 column 1 (char 0)
Try processing row 31
Row 31: retrying due to error: Expecting value: line 1 column 1 (char 0)
Try processing row 31
Row 31: re